# khwab — the cutting room, in Colab

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/zistgah/khwab/blob/main/khwab.ipynb)

A reel goes in, a published book comes out. Same engine as the local app; the gates are unchanged.

In [ ]:
#@title 1 · Fetch khwab  { display-mode: "form" }
import os, shutil, urllib.request, importlib.util
BASE = "https://raw.githubusercontent.com/zistgah/khwab/main/"  #@param {type:"string"}
for f in ("khwab.py", "seed.tmpl.sh", "docs/studio.html", "docs/js/cycler.js", "docs/js/khwab.js",
          "readers/media.js", "overlays/CONTRACT.md", "overlays/CONTEXT.md"):
    os.makedirs(os.path.dirname(f) or ".", exist_ok=True)
    if not os.path.exists(f):
        try: urllib.request.urlretrieve(BASE + f, f); print("fetched", f)
        except Exception as e: print("could not fetch", f, "-", e)
os.environ["KHWAB_HOME"] = "/content/khwab-projects"
for t in ("git", "gh", "misty"):
    print("%-6s %s" % (t, shutil.which(t) or "ABSENT — declared, not worked around"))

In [ ]:
#@title 2 · Open the cutting room right here  { display-mode: "form" }
# The composing part is a static page: it runs in the notebook output, in this browser, offline.
from IPython.display import IFrame, HTML, display
import base64, pathlib
h = pathlib.Path("docs/studio.html").read_text()
# inline the two modules so the iframe needs no server
for m in ("cycler.js", "khwab.js"):
    src = pathlib.Path("docs/js/%s" % m).read_text()
    h = h.replace("./js/%s" % m, "data:text/javascript;base64," + base64.b64encode(src.encode()).decode())
pathlib.Path("studio-inline.html").write_text(h)
display(HTML('<iframe srcdoc="%s" style="width:100%%;height:820px;border:0"></iframe>'
             % h.replace('"', "&quot;")))
print("Drop your reel above. Export the payload when the lockup passes, then run cell 3.")

In [ ]:
#@title 3 · Import the payload you exported  { display-mode: "form" }
SLUG = "khwab"  #@param {type:"string"}
PAYLOAD = "/content/khwab-payload.zip"  #@param {type:"string"}
import importlib.util, json, sys
spec = importlib.util.spec_from_file_location("khwab", "khwab.py")
K = importlib.util.module_from_spec(spec); spec.loader.exec_module(K)
K.HOME = os.environ["KHWAB_HOME"]
print(json.dumps(K.cmd_import(SLUG, PAYLOAD), indent=2))

In [ ]:
#@title 4 · Install gh and misty  { display-mode: "form" }
!type -p gh >/dev/null || (curl -fsSL https://cli.github.com/packages/githubcli-archive-keyring.gpg \
  | sudo dd of=/usr/share/keyrings/githubcli-archive-keyring.gpg >/dev/null 2>&1 \
 && echo "deb [signed-by=/usr/share/keyrings/githubcli-archive-keyring.gpg] https://cli.github.com/packages stable main" \
  | sudo tee /etc/apt/sources.list.d/github-cli.list >/dev/null \
 && sudo apt-get update -qq && sudo apt-get install -y -qq gh)
!pip -q install misty-doi
!type -p gh && gh --version | head -1

In [ ]:
#@title 5 · Authenticate  { display-mode: "form" }
from getpass import getpass
import os, pathlib
if not os.environ.get("GH_TOKEN"): os.environ["GH_TOKEN"] = getpass("GitHub token (repo scope): ")
!echo "$GH_TOKEN" | gh auth login --with-token && gh auth status
zp = "/content/zenodo_token"
if not os.path.exists(zp):
    pathlib.Path(zp).write_text(getpass("Zenodo token: ").strip()); os.chmod(zp, 0o600)
os.environ["ZENODO_TOKEN_PATH"] = zp
print("zenodo token at", zp)

In [ ]:
#@title 6 · Build, then stage  { display-mode: "form" }
import json
print(json.dumps(K.cmd_build(SLUG), indent=2))
p = K.cmd_run(SLUG, "stage")
for line in p.stdout: print(line, end="")
print("exit:", p.wait())

In [ ]:
#@title 7 · Push  { display-mode: "form" }
w = input('Type the gate word exactly ("PUSH seed"), or anything else to abort: ')
p = K.cmd_run(SLUG, "push", word=w)
for line in p.stdout: print(line, end="")
print("exit:", p.wait(), " (3 = gate refused, nothing done)")

In [ ]:
#@title 8 · Mint — permanent  { display-mode: "form" }
OVERRIDE_REHEARSAL = True  #@param {type:"boolean"}
w1 = input('PUSH gate word: ')
w2 = input('MINT gate word: ')
if w2.strip() != "MINT " + SLUG:
    print("mint word does not match — not running.")
else:
    p = K.cmd_run(SLUG, "mint", word=w1, override=OVERRIDE_REHEARSAL)
    for line in p.stdout: print(line, end="")
    print("exit:", p.wait())

---
The reel is minted **with** the book: one file, one hash, one deposit. Chapters are cues into it,
not cut copies.

Colab VMs are ephemeral — set `KHWAB_HOME` to a Drive path in cell 1 if you want the project to
survive the session.